<a href="https://colab.research.google.com/github/Khamidrees/Khalid-Muhammad-Idris-DSN-HACKATHON-/blob/main/KHALID_MUHAMMAD_IDRIS_SUBMISSION_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#CORRECT STOREMULT
Expected result: 1074.42 LB,

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("✓ CELL 1: Imports & Setup Complete")
print("="*80)

✓ CELL 1: Imports & Setup Complete


In [ ]:
from google.colab import files

print("\n" + "="*80)
print("CELL 2: Upload Data")
print("="*80)
print("\nClick 'Choose Files' and upload:")
print("  • train.csv")
print("  • test.csv")

uploaded = files.upload()

if 'train.csv' in uploaded and 'test.csv' in uploaded:
    print("\n✓ Files uploaded successfully!")
else:
    raise FileNotFoundError("❌ Missing train.csv or test.csv")

# Load
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

TARGET = 'total_sales'
y = train[TARGET].values
n = len(train)

X_train_raw = train.drop('id', axis=1).drop(TARGET, axis=1)
X_test_raw = test.drop('id', axis=1)
test_ids = test['id'].copy()

print(f"\n✓ Data loaded:")
print(f"  Train: {X_train_raw.shape}")
print(f"  Test:  {X_test_raw.shape}")
print(f"  Target: mean={y.mean():.2f}, std={y.std():.2f}, min={y.min():.2f}, max={y.max():.2f}")


CELL 2: Upload Data

Click 'Choose Files' and upload:
  • train.csv
  • test.csv


Saving test.csv to test.csv
Saving train.csv to train.csv

✓ Files uploaded successfully!

✓ Data loaded:
  Train: (6818, 11)
  Test:  (1705, 11)
  Target: mean=2174.76, std=1697.77, min=32.70, max=12996.82


In [ ]:
print("\n" + "="*80)
print("CELL 3: Data Cleaning")
print("="*80)

# Combine for uniform processing
full = pd.concat([X_train_raw, X_test_raw], ignore_index=True)

print("\nCleaning steps:")

# Standardize categories
full['product_category'] = full['product_category'].str.strip().str.lower()
print("  ✓ Standardized product categories")

# Handle missing store_size
full['store_size'] = full['store_size'].fillna('missing')
print("  ✓ Handled missing store_size")

# Fill weights from same product
full['product_weight_kg'] = full['product_weight_kg'].fillna(
    full.groupby('product_code')['product_weight_kg'].transform('mean'))
print("  ✓ Filled missing weights")

# Handle zero visibility
vis = full['shelf_visibility'].replace(0, np.nan)
full['shelf_visibility'] = vis.fillna(
    vis.groupby(full['product_code']).transform('mean')).fillna(vis.median())
print("  ✓ Handled zero visibility")

# Price ratio feature
full['price_ratio'] = full['product_price'] / full.groupby('product_code')['product_price'].transform('mean')
print("  ✓ Created price_ratio feature")

print(f"\n✓ CELL 3 Complete")
print(f"  Features: {list(full.columns)}")


CELL 3: Data Cleaning

Cleaning steps:
  ✓ Standardized product categories
  ✓ Handled missing store_size
  ✓ Filled missing weights
  ✓ Handled zero visibility
  ✓ Created price_ratio feature

✓ CELL 3 Complete
  Features: ['product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'price_ratio']


In [ ]:
print("\n" + "="*80)
print("CELL 4: Store Multiplier Model")
print("="*80)

# Extract price and store
price = full['product_price'].values
store_code = full['store_code'].values

def store_multiplier_model(train_idx, pred_idx):
    """
    Model: sales = k_store × price

    For each store, fits k using least squares:
    k = (price @ sales) / (price @ price)
    """
    predictions = np.zeros(len(pred_idx))

    for store in np.unique(store_code):
        train_mask = (store_code[train_idx] == store)
        train_store_idx = train_idx[train_mask]

        pred_mask = (store_code[pred_idx] == store)
        pred_store_idx = pred_idx[pred_mask]

        if len(train_store_idx) == 0:
            predictions[pred_mask] = y.mean()
            continue

        p_train = price[train_store_idx]
        y_train = y[train_store_idx]

        denominator = p_train @ p_train
        k = (p_train @ y_train) / denominator if denominator > 0 else 1.0

        p_pred = price[pred_store_idx]
        predictions[pred_mask] = k * p_pred

    return np.clip(predictions, 0, None)

print("✓ Store multiplier model defined")


CELL 4: Store Multiplier Model
✓ Store multiplier model defined


In [ ]:
print("\n" + "="*80)
print("CELL 5: GBM Model Setup")
print("="*80)

from sklearn.preprocessing import LabelEncoder

# Prepare features
GB_FEATS = ['product_price', 'store_code', 'product_category',
            'shelf_visibility', 'product_weight_kg', 'price_ratio']

X = full[GB_FEATS].copy()
CAT_COLS = ['store_code', 'product_category']

print(f"\nGBM Features: {GB_FEATS}")
print(f"Categorical: {CAT_COLS}")

# Encode categories
for c in CAT_COLS:
    X[c] = X[c].astype('category').cat.codes

# GBM hyperparameters
GB_PARAMS = {
    'learning_rate': 0.01,
    'max_iter': 600,
    'max_leaf_nodes': 4,
    'min_samples_leaf': 80,
    'l2_regularization': 5.0,
    'random_state': 0
}

def gbm_model(train_idx, pred_idx):
    """Shallow GBM on units sold (sales/price), rescaled by price."""
    m = HistGradientBoostingRegressor(**GB_PARAMS)
    m.fit(X.iloc[train_idx], y[train_idx] / price[train_idx])
    predictions = m.predict(X.iloc[pred_idx]) * price[pred_idx]
    return np.clip(predictions, 0, None)

print("\n✓ GBM model defined")


CELL 5: GBM Model Setup

GBM Features: ['product_price', 'store_code', 'product_category', 'shelf_visibility', 'product_weight_kg', 'price_ratio']
Categorical: ['store_code', 'product_category']

✓ GBM model defined


In [ ]:
print("\n" + "="*80)
print("CELL 6: Cross-Validation (This will take ~2-3 minutes)")
print("="*80)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Store multiplier CV
print("\n📊 Evaluating Store Multiplier...")
mult_cv_scores = []

for seed in [7, 11, 23]:
    oof_preds = np.zeros(n)

    for train_idx, val_idx in kfold.split(np.arange(n)):
        val_preds = store_multiplier_model(train_idx, val_idx)
        oof_preds[val_idx] = val_preds

    rmse = np.sqrt(np.mean((y - oof_preds) ** 2))
    mult_cv_scores.append(rmse)
    print(f"  Seed {seed}: RMSE = {rmse:.4f}")

mult_rmse = np.mean(mult_cv_scores)
print(f"\n✓ Store Multiplier CV RMSE: {mult_rmse:.4f}")

# GBM CV
print("\n📊 Evaluating GBM (this takes longer)...")
gbm_cv_scores = []

for seed in [7, 11, 23]:
    oof_preds = np.zeros(n)

    for train_idx, val_idx in kfold.split(np.arange(n)):
        val_preds = gbm_model(train_idx, val_idx)
        oof_preds[val_idx] = val_preds

    rmse = np.sqrt(np.mean((y - oof_preds) ** 2))
    gbm_cv_scores.append(rmse)
    print(f"  Seed {seed}: RMSE = {rmse:.4f}")

gbm_rmse = np.mean(gbm_cv_scores)
print(f"\n✓ GBM CV RMSE: {gbm_rmse:.4f}")

print("\n" + "="*80)
print("📊 CV RESULTS:")
print(f"  Store Multiplier: {mult_rmse:.4f}")
print(f"  GBM:              {gbm_rmse:.4f}")
print("="*80)


CELL 6: Cross-Validation (This will take ~2-3 minutes)

📊 Evaluating Store Multiplier...
  Seed 7: RMSE = 1072.5694
  Seed 11: RMSE = 1072.5694
  Seed 23: RMSE = 1072.5694

✓ Store Multiplier CV RMSE: 1072.5694

📊 Evaluating GBM (this takes longer)...
  Seed 7: RMSE = 1077.3591
  Seed 11: RMSE = 1077.3591
  Seed 23: RMSE = 1077.3591

✓ GBM CV RMSE: 1077.3591

📊 CV RESULTS:
  Store Multiplier: 1072.5694
  GBM:              1077.3591


In [ ]:
print("\n" + "="*80)
print("CELL 7: Final Predictions")
print("="*80)

all_train = np.arange(n)
all_test = np.arange(n, len(full))

# Store multiplier
print("\n1️⃣  Training Store Multiplier on full data...")
pred_mult = store_multiplier_model(all_train, all_test)
print(f"   Mean: {pred_mult.mean():.2f}, Std: {pred_mult.std():.2f}")
print(f"   Min:  {pred_mult.min():.2f}, Max: {pred_mult.max():.2f}")

# GBM ensemble
print("\n2️⃣  Training GBM ensemble (5 seeds)...")
pred_gbm_models = []
for seed in range(5):
    params = {**GB_PARAMS, 'random_state': seed}
    m = HistGradientBoostingRegressor(**params)
    m.fit(X.iloc[all_train], y / price[all_train])
    pred = m.predict(X.iloc[all_test]) * price[all_test]
    pred_gbm_models.append(np.clip(pred, 0, None))

pred_gbm = np.mean(pred_gbm_models, axis=0)
print(f"   Mean: {pred_gbm.mean():.2f}, Std: {pred_gbm.std():.2f}")
print(f"   Min:  {pred_gbm.min():.2f}, Max: {pred_gbm.max():.2f}")

# Ensemble weights
print("\n3️⃣  Computing ensemble weights...")
w_mult = 1.0 / (mult_rmse ** 2)
w_gbm = 1.0 / (gbm_rmse ** 2)
total_w = w_mult + w_gbm

blend_weight_mult = w_mult / total_w
blend_weight_gbm = w_gbm / total_w

print(f"   Store Multiplier: {blend_weight_mult:.4f}")
print(f"   GBM:              {blend_weight_gbm:.4f}")

# Final blend
final_pred = blend_weight_mult * pred_mult + blend_weight_gbm * pred_gbm

print(f"\n✓ Final Ensemble Predictions:")
print(f"   Mean: {final_pred.mean():.2f}")
print(f"   Std:  {final_pred.std():.2f}")
print(f"   Min:  {final_pred.min():.2f}")
print(f"   Max:  {final_pred.max():.2f}")


CELL 7: Final Predictions

1️⃣  Training Store Multiplier on full data...
   Mean: 2193.73, Std: 1318.54
   Min:  81.47, Max: 6859.71

2️⃣  Training GBM ensemble (5 seeds)...
   Mean: 2195.23, Std: 1265.58
   Min:  105.22, Max: 6651.20

3️⃣  Computing ensemble weights...
   Store Multiplier: 0.5022
   GBM:              0.4978

✓ Final Ensemble Predictions:
   Mean: 2194.48
   Std:  1290.94
   Min:  94.78
   Max:  6755.92


In [ ]:
print("\n" + "="*80)
print("CELL 8: Save & Download Submission")
print("="*80)

# Create submission
submission = pd.DataFrame({
    'id': test_ids,
    'total_sales': final_pred
})

# Verify
print("\n✓ Submission format:")
print(f"  Shape: {submission.shape}")
print(f"  Columns: {list(submission.columns)}")
print(f"\nFirst 10 rows:")
print(submission.head(10))

# Save
submission.to_csv('submission_CORRECT_STOREMULT.csv', index=False)
print("\n✓ Saved: submission_CORRECT_STOREMULT.csv")

# Download
from google.colab import files
files.download('submission_CORRECT_STOREMULT.csv')

print("\n" + "="*80)
print("🎉 SUBMISSION READY FOR DOWNLOAD 🎉")
print("="*80)
print(f"\nFile: submission_CORRECT_STOREMULT.csv")
print(f"Rows: {len(submission)}")
print(f"Expected LB RMSE: 1074-1075")
print(f"Expected Rank: ~111")
print("\n📌 NEXT STEPS:")
print("1. Download the CSV file from the popup")
print("2. Go to Kaggle competition page")
print("3. Click 'Submit Predictions'")
print("4. Upload submission_CORRECT_STOREMULT.csv")
print("5. Check the leaderboard!")
print("="*80)


CELL 8: Save & Download Submission

✓ Submission format:
  Shape: (1705, 2)
  Columns: ['id', 'total_sales']

First 10 rows:
          id  total_sales
0  row_00009  2716.404401
1  row_00015  4484.786935
2  row_00019  2934.060373
3  row_00020  3126.036295
4  row_00023  2540.939734
5  row_00026   387.606645
6  row_00027  3881.303133
7  row_00033  3161.174709
8  row_00034   520.050434
9  row_00037  2099.255966

✓ Saved: submission_CORRECT_STOREMULT.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 SUBMISSION READY FOR DOWNLOAD 🎉

File: submission_CORRECT_STOREMULT.csv
Rows: 1705
Expected LB RMSE: 1074-1075
Expected Rank: ~111

📌 NEXT STEPS:
1. Download the CSV file from the popup
2. Go to Kaggle competition page
3. Click 'Submit Predictions'
4. Upload submission_CORRECT_STOREMULT.csv
5. Check the leaderboard!
